In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [16]:
# 데이터 로드
bank_df = pd.read_csv("../JeonJongHyeok/data/df_encoded_final.csv")

bank_df = bank_df.drop('trans_ct_bin', axis=1)

In [17]:
# 목표 데이터 분리
Y = bank_df["churn"]

print(Y)

0        1
1        1
2        1
3        1
4        1
        ..
10122    1
10123    0
10124    0
10125    0
10126    0
Name: churn, Length: 10127, dtype: int64


In [18]:
# feature 데이터 분리
X = bank_df[[k for k in bank_df.columns if k != "churn"]]

print(X)

       age  gender  dependents  relationship_months  product_count  \
0       45       0           3                   39              5   
1       49       1           5                   44              6   
2       51       0           3                   36              4   
3       40       1           4                   34              3   
4       40       0           3                   21              5   
...    ...     ...         ...                  ...            ...   
10122   50       0           2                   40              3   
10123   41       0           2                   25              4   
10124   44       1           1                   36              5   
10125   30       0           2                   36              4   
10126   43       1           2                   25              6   

       inactive_months  contact_count  credit_limit  revolving_balance  \
0                    1              3       12691.0                777   
1          

In [19]:
from sklearn.model_selection import train_test_split

# train - test split을 활용해서 학습/테스트 데이터 분리
(X_train, X_test, y_train, y_test) = train_test_split(
    X,
    Y,
    test_size=0.2,
    stratify=Y,
    random_state=42
)

print(X_train)

      age  gender  dependents  relationship_months  product_count  \
1602   54       1           3                   49              6   
7791   51       0           0                   45              3   
7177   45       1           4                   29              3   
97     53       0           3                   35              5   
4820   48       1           2                   40              3   
...   ...     ...         ...                  ...            ...   
6821   57       1           3                   39              6   
6178   48       1           2                   27              3   
2544   50       1           1                   36              6   
4945   43       0           5                   36              3   
3640   54       0           2                   36              5   

      inactive_months  contact_count  credit_limit  revolving_balance  \
1602                2              3       13184.0                  0   
7791                2    

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import HistGradientBoostingClassifier

# 테스트용 모델 학습
hgb_clf = HistGradientBoostingClassifier(
    random_state=42,
    max_iter=5000,    # 약학습기 개수
    validation_fraction=0.2, # 20%를 검증용 데이터로 사용
    early_stopping=True, # 평가지표, tol 기준
    n_iter_no_change=10, # 충분히 멈춰도 되는 지점이 오더라도 일정 횟수 더 학습하라는 의미
)

# 결과 출력
results = cross_validate(hgb_clf, X_train, y_train, cv=5, scoring='accuracy', return_train_score=True, n_jobs=-1, verbose=1)
results_df = pd.DataFrame(results)
print(results_df)
print("train score:", results_df['train_score'].mean())
print("test score:", results_df['test_score'].mean())

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 18 concurrent workers.


   fit_time  score_time  test_score  train_score
0  1.707045    0.075944    0.963603     0.993981
1  1.741242    0.047821    0.978395     0.993828
2  1.625737    0.079101    0.967284     0.995062
3  2.345122    0.079390    0.968519     0.994908
4  1.996003    0.037145    0.963580     0.994754
train score: 0.9945068347934418
test score: 0.9682760984303244


[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   20.2s finished


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(random_state=42, early_stopping=True)

# 최적의 경우를 찾고자 하는 하이퍼 파라미터들을 dictionary 형태로 준비
params = {
    'max_iter':[1000, 3000, 5000],    # 약학습기 개수
    'validation_fraction':[0.1, 0.2, 0.3], # 20%를 검증용 데이터로 사용
    'n_iter_no_change':[5, 10, 20], # 충분히 멈춰도 되는 지점이 오더라도 일정 횟수 더 학습하라는 의미
}

# GridSearchCV 적용
grid_search = GridSearchCV(model, param_grid=params, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# 어떤 parameter를 전달했을 때 가장 좋은 결과를 얻을 수 있는지
print(grid_search.best_params_)
# 그때의 score
print(grid_search.best_score_)
# 해당 parameter를 전달한 학습 모델
print(grid_search.best_estimator_)

{'max_iter': 1000, 'n_iter_no_change': 20, 'validation_fraction': 0.3}
0.9712381474627001
HistGradientBoostingClassifier(early_stopping=True, max_iter=1000,
                               n_iter_no_change=20, random_state=42,
                               validation_fraction=0.3)


In [22]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import HistGradientBoostingClassifier

# 1차 하이퍼 파라미터 적용
hgb_clf = HistGradientBoostingClassifier(
    random_state=42,
    max_iter=1000,    # 약학습기 개수
    validation_fraction=0.3, # 20%를 검증용 데이터로 사용
    early_stopping=True, # 평가지표, tol 기준
    n_iter_no_change=20, # 충분히 멈춰도 되는 지점이 오더라도 일정 횟수 더 학습하라는 의미
)

# 결과 출력
results = cross_validate(hgb_clf, X_train, y_train, cv=5, scoring='accuracy', return_train_score=True, n_jobs=-1, verbose=1)
results_df = pd.DataFrame(results)
print(results_df)
print("train score:", results_df['train_score'].mean())
print("test score:", results_df['test_score'].mean())

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 18 concurrent workers.


   fit_time  score_time  test_score  train_score
0  2.404779    0.105120    0.971006     0.990895
1  2.500837    0.017848    0.975926     0.992285
2  2.719761    0.023517    0.970988     0.991822
3  3.249553    0.103066    0.972222     0.992594
4  2.933834    0.082105    0.966049     0.990434
train score: 0.9916059526326718
test score: 0.9712381474627001


[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   24.7s finished


In [24]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(random_state=42, early_stopping=True)

# 2차 하이퍼 파라미터 탐색
params = {
    'max_iter':[800, 1000, 1200],    # 약학습기 개수
    'validation_fraction':[0.1, 0.2, 0.3], # 20%를 검증용 데이터로 사용
    'n_iter_no_change':[15, 20, 25], # 충분히 멈춰도 되는 지점이 오더라도 일정 횟수 더 학습하라는 의미
}

# GridSearchCV 적용
grid_search = GridSearchCV(model, param_grid=params, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# 어떤 parameter를 전달했을 때 가장 좋은 결과를 얻을 수 있는지
print(grid_search.best_params_)
# 그때의 score
print(grid_search.best_score_)
# 해당 parameter를 전달한 학습 모델
print(grid_search.best_estimator_)

{'max_iter': 800, 'n_iter_no_change': 25, 'validation_fraction': 0.1}
0.9713621373789995
HistGradientBoostingClassifier(early_stopping=True, max_iter=800,
                               n_iter_no_change=25, random_state=42)


In [25]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import HistGradientBoostingClassifier

# 2차 하이퍼 파라미터 적용
hgb_clf = HistGradientBoostingClassifier(
    random_state=42,
    max_iter=800,    # 약학습기 개수
    validation_fraction=0.1, # 20%를 검증용 데이터로 사용
    early_stopping=True, # 평가지표, tol 기준
    n_iter_no_change=25, # 충분히 멈춰도 되는 지점이 오더라도 일정 횟수 더 학습하라는 의미
)

# 결과 출력
results = cross_validate(hgb_clf, X_train, y_train, cv=5, scoring='accuracy', return_train_score=True, n_jobs=-1, verbose=1)
results_df = pd.DataFrame(results)
print(results_df)
print("train score:", results_df['train_score'].mean())
print("test score:", results_df['test_score'].mean())

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 18 concurrent workers.


   fit_time  score_time  test_score  train_score
0  2.212660    0.039134    0.966687     0.995833
1  3.336704    0.116408    0.975926     0.997223
2  3.765055    0.087923    0.975309     0.997223
3  3.786807    0.036550    0.969753     0.997068
4  3.201654    0.033357    0.969136     0.997840
train score: 0.9970373656328755
test score: 0.9713621373789995


[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   23.9s finished


In [26]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(random_state=42, early_stopping=True)

# 2차 하이퍼 파라미터 탐색
params = {
    'max_iter':[300, 500, 800],    # 약학습기 개수
    'validation_fraction':[0.1, 0.15, 0.2], # 20%를 검증용 데이터로 사용
    'n_iter_no_change':[20, 25, 30], # 충분히 멈춰도 되는 지점이 오더라도 일정 횟수 더 학습하라는 의미
}

# GridSearchCV 적용
grid_search = GridSearchCV(model, param_grid=params, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# 어떤 parameter를 전달했을 때 가장 좋은 결과를 얻을 수 있는지
print(grid_search.best_params_)
# 그때의 score
print(grid_search.best_score_)
# 해당 parameter를 전달한 학습 모델
print(grid_search.best_estimator_)

{'max_iter': 300, 'n_iter_no_change': 30, 'validation_fraction': 0.1}
0.9717325839102522
HistGradientBoostingClassifier(early_stopping=True, max_iter=300,
                               n_iter_no_change=30, random_state=42)


In [27]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(random_state=42, early_stopping=True)

# 2차 하이퍼 파라미터 탐색
params = {
    'max_iter':[100, 200, 300],    # 약학습기 개수
    'validation_fraction':[0.1, 0.15, 0.2], # 20%를 검증용 데이터로 사용
    'n_iter_no_change':[25, 30, 35], # 충분히 멈춰도 되는 지점이 오더라도 일정 횟수 더 학습하라는 의미
}

# GridSearchCV 적용
grid_search = GridSearchCV(model, param_grid=params, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# 어떤 parameter를 전달했을 때 가장 좋은 결과를 얻을 수 있는지
print(grid_search.best_params_)
# 그때의 score
print(grid_search.best_score_)
# 해당 parameter를 전달한 학습 모델
print(grid_search.best_estimator_)

{'max_iter': 200, 'n_iter_no_change': 35, 'validation_fraction': 0.1}
0.9719792690078523
HistGradientBoostingClassifier(early_stopping=True, max_iter=200,
                               n_iter_no_change=35, random_state=42)


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(random_state=42, early_stopping=True, validation_fraction=0.1)

# 2차 하이퍼 파라미터 탐색
params = {
    'max_iter':[150, 200, 250],    # 약학습기 개수
    'n_iter_no_change':[25, 30, 35], # 충분히 멈춰도 되는 지점이 오더라도 일정 횟수 더 학습하라는 의미
}

# GridSearchCV 적용
grid_search = GridSearchCV(model, param_grid=params, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# 어떤 parameter를 전달했을 때 가장 좋은 결과를 얻을 수 있는지
print(grid_search.best_params_)
# 그때의 score
print(grid_search.best_score_)
# 해당 parameter를 전달한 학습 모델
print(grid_search.best_estimator_)

{'max_iter': 200}
0.9729662378808996
HistGradientBoostingClassifier(early_stopping=True, max_iter=200,
                               n_iter_no_change=15, random_state=42)


In [18]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(
    random_state=42,
    early_stopping=True,
    n_iter_no_change=15,
    validation_fraction=0.1
)

# 2차 하이퍼 파라미터 탐색
params = {
    'max_iter':[50, 100, 200],    # 약학습기 개수
}

# GridSearchCV 적용
grid_search = GridSearchCV(model, param_grid=params, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# 어떤 parameter를 전달했을 때 가장 좋은 결과를 얻을 수 있는지
print(grid_search.best_params_)
# 그때의 score
print(grid_search.best_score_)
# 해당 parameter를 전달한 학습 모델
print(grid_search.best_estimator_)

{'max_iter': 100}
0.9729662378808996
HistGradientBoostingClassifier(early_stopping=True, n_iter_no_change=15,
                               random_state=42)
